# Security Analysis with AI

**Category:** 08-InfoSec  
**Description:** AI-assisted security analysis and vulnerability assessment  
**Use Cases:** Log analysis, threat detection, security audits

---

## What You'll Learn

1. Analyze security logs with AI
2. Detect anomalous patterns
3. Classify security events
4. Generate security reports
5. AI-powered threat assessment

---

In [ ]:
# =============================================================================
# DEPENDENCIES & MODEL ALIASES
# =============================================================================
%pip install -q pandas numpy matplotlib seaborn

# Model aliases
CLAUDE = "anthropic-chat:claude-sonnet-4-5-20250929"
GPT = "openai-chat:gpt-5"
GEMINI = "gemini:gemini-2.5-flash"

MODEL = CLAUDE

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')

---

# Part 1: Generate Sample Security Logs

For demonstration, we'll create synthetic security logs.

---

In [ ]:
def generate_security_logs(n_events=1000):
    """Generate synthetic security log data."""
    np.random.seed(42)
    
    event_types = [
        'LOGIN_SUCCESS', 'LOGIN_FAILURE', 'LOGOUT',
        'FILE_ACCESS', 'FILE_MODIFY', 'FILE_DELETE',
        'NETWORK_CONNECTION', 'FIREWALL_BLOCK',
        'PRIVILEGE_ESCALATION', 'SUSPICIOUS_ACTIVITY'
    ]
    
    users = ['admin', 'jsmith', 'mjohnson', 'alee', 'bwilson', 
             'cgarcia', 'dkim', 'system', 'unknown']
    
    source_ips = [
        '192.168.1.100', '192.168.1.101', '192.168.1.102',
        '10.0.0.50', '10.0.0.51', '10.0.0.52',
        '203.0.113.45', '198.51.100.23', '172.16.0.1'
    ]
    
    severities = ['INFO', 'WARNING', 'ERROR', 'CRITICAL']
    
    # Generate events
    events = []
    base_time = datetime.now() - timedelta(days=7)
    
    for i in range(n_events):
        # Time with some clustering
        time_offset = np.random.exponential(scale=600)  # seconds
        timestamp = base_time + timedelta(seconds=time_offset * (i+1) / 10)
        
        event_type = np.random.choice(event_types, p=[
            0.25, 0.10, 0.15, 0.20, 0.08, 0.02, 0.10, 0.05, 0.02, 0.03
        ])
        
        user = np.random.choice(users)
        source_ip = np.random.choice(source_ips)
        
        # Severity based on event type
        if event_type in ['PRIVILEGE_ESCALATION', 'SUSPICIOUS_ACTIVITY']:
            severity = np.random.choice(['WARNING', 'ERROR', 'CRITICAL'], p=[0.3, 0.4, 0.3])
        elif event_type in ['LOGIN_FAILURE', 'FIREWALL_BLOCK']:
            severity = np.random.choice(['INFO', 'WARNING', 'ERROR'], p=[0.3, 0.5, 0.2])
        else:
            severity = np.random.choice(severities, p=[0.7, 0.2, 0.08, 0.02])
        
        events.append({
            'timestamp': timestamp,
            'event_type': event_type,
            'user': user,
            'source_ip': source_ip,
            'severity': severity,
            'event_id': f'EVT-{i+1:06d}'
        })
    
    return pd.DataFrame(events)

# Generate logs
logs = generate_security_logs(1000)
print(f"Generated {len(logs)} security events")
logs.head(10)

---

# Part 2: Security Event Analysis

---

In [ ]:
# Event distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Events by type
event_counts = logs['event_type'].value_counts()
axes[0, 0].barh(event_counts.index, event_counts.values, color='steelblue')
axes[0, 0].set_title('Events by Type')
axes[0, 0].set_xlabel('Count')

# 2. Severity distribution
severity_order = ['INFO', 'WARNING', 'ERROR', 'CRITICAL']
severity_counts = logs['severity'].value_counts().reindex(severity_order)
colors = ['green', 'yellow', 'orange', 'red']
axes[0, 1].pie(severity_counts.values, labels=severity_counts.index, 
               autopct='%1.1f%%', colors=colors, startangle=90)
axes[0, 1].set_title('Severity Distribution')

# 3. Events by user
user_counts = logs['user'].value_counts().head(10)
axes[1, 0].bar(user_counts.index, user_counts.values, color='coral')
axes[1, 0].set_title('Events by User')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Events by source IP
ip_counts = logs['source_ip'].value_counts().head(10)
axes[1, 1].barh(ip_counts.index, ip_counts.values, color='purple')
axes[1, 1].set_title('Events by Source IP')

plt.tight_layout()
plt.show()

In [ ]:
# Timeline analysis
logs['hour'] = logs['timestamp'].dt.hour
logs['day'] = logs['timestamp'].dt.day_name()

# Hourly distribution
hourly = logs.groupby('hour').size()

plt.figure(figsize=(12, 5))
plt.bar(hourly.index, hourly.values, color='steelblue')
plt.xlabel('Hour of Day')
plt.ylabel('Event Count')
plt.title('Security Events by Hour')
plt.xticks(range(24))
plt.show()

---

# Part 3: Threat Detection

---

In [ ]:
# Detect potential brute force attacks (multiple failed logins)
def detect_brute_force(logs, threshold=5, window_minutes=10):
    """Detect potential brute force attacks."""
    failed_logins = logs[logs['event_type'] == 'LOGIN_FAILURE'].copy()
    failed_logins = failed_logins.sort_values('timestamp')
    
    alerts = []
    
    # Group by source IP
    for ip, group in failed_logins.groupby('source_ip'):
        group = group.sort_values('timestamp')
        
        # Rolling window count
        for i, row in group.iterrows():
            window_start = row['timestamp'] - timedelta(minutes=window_minutes)
            window_events = group[
                (group['timestamp'] >= window_start) & 
                (group['timestamp'] <= row['timestamp'])
            ]
            
            if len(window_events) >= threshold:
                alerts.append({
                    'alert_type': 'BRUTE_FORCE',
                    'source_ip': ip,
                    'timestamp': row['timestamp'],
                    'failed_attempts': len(window_events),
                    'users_targeted': window_events['user'].unique().tolist()
                })
    
    return pd.DataFrame(alerts).drop_duplicates(subset=['source_ip', 'timestamp'])

brute_force_alerts = detect_brute_force(logs)
print(f"Detected {len(brute_force_alerts)} potential brute force attacks")
if len(brute_force_alerts) > 0:
    print(brute_force_alerts)

In [ ]:
# Detect privilege escalation patterns
priv_esc = logs[logs['event_type'] == 'PRIVILEGE_ESCALATION']

print(f"Privilege Escalation Events: {len(priv_esc)}")
print("\nBy User:")
print(priv_esc['user'].value_counts())
print("\nBy Source IP:")
print(priv_esc['source_ip'].value_counts())

In [ ]:
# Suspicious activity summary
suspicious = logs[logs['event_type'] == 'SUSPICIOUS_ACTIVITY']
high_severity = logs[logs['severity'].isin(['ERROR', 'CRITICAL'])]

print("=" * 50)
print("SECURITY SUMMARY")
print("=" * 50)
print(f"Total Events: {len(logs)}")
print(f"Failed Logins: {len(logs[logs['event_type'] == 'LOGIN_FAILURE'])}")
print(f"Privilege Escalations: {len(priv_esc)}")
print(f"Suspicious Activities: {len(suspicious)}")
print(f"High Severity Events: {len(high_severity)}")
print(f"Firewall Blocks: {len(logs[logs['event_type'] == 'FIREWALL_BLOCK'])}")

---

# Part 4: AI-Powered Security Analysis

---

In [ ]:
%%ai $MODEL
Analyze this security log summary and provide threat assessment:

Security Event Summary (7 days):
- Total Events: 1,000
- Failed Logins: 100 (10%)
- Privilege Escalations: 20 (2%)
- Suspicious Activities: 30 (3%)
- High Severity Events: 100 (10%)
- Firewall Blocks: 50 (5%)

Top source IPs for failed logins:
- 203.0.113.45 (external): 25 attempts
- 198.51.100.23 (external): 18 attempts
- 192.168.1.100 (internal): 12 attempts

Please provide:
1. Overall security posture assessment
2. Potential attack patterns identified
3. Recommended immediate actions
4. Long-term security improvements
5. Risk rating (Critical/High/Medium/Low)

---

# Part 5: Security Heatmap

---

In [ ]:
# Create security event heatmap (hour vs day)
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = logs.pivot_table(
    index='day', 
    columns='hour', 
    values='event_id', 
    aggfunc='count'
).reindex(day_order)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, cbar_kws={'label': 'Event Count'})
plt.title('Security Events Heatmap (Day vs Hour)')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

In [ ]:
# Event type by severity heatmap
type_severity = logs.pivot_table(
    index='event_type',
    columns='severity',
    values='event_id',
    aggfunc='count'
).fillna(0).reindex(columns=severity_order)

plt.figure(figsize=(10, 8))
sns.heatmap(type_severity, cmap='Reds', annot=True, fmt='g', cbar_kws={'label': 'Count'})
plt.title('Event Type vs Severity')
plt.tight_layout()
plt.show()

---

## Summary

This notebook demonstrated:

1. **Log Generation** - Creating synthetic security logs
2. **Event Analysis** - Distribution by type, severity, user, IP
3. **Threat Detection** - Brute force and privilege escalation
4. **AI Analysis** - Security assessment and recommendations
5. **Visualization** - Heatmaps and dashboards

---

**Next:** Try the InfoSec pentest notebook in `08-InfoSec/` from CalliopeLabExamples